# FlowDistill MRI — Colab interface
This notebook is a thin interface over the repository CLI. Long-running actions are opt-in.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/hugoaslm/flowdistill-mri.git'
REPO_ROOT = Path('/content/flowdistill-mri')
if not (REPO_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyyaml>=6', 'safetensors>=0.4'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
print('Repository:', REPO_ROOT)

In [ ]:
import platform

import torch

print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})

## Optional persistent Google Drive outputs

In [ ]:
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = Path('/content/drive/MyDrive/flowdistill-mri/runs')
else:
    RUN_ROOT = Path('/content/flowdistill-mri-runs')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Run root:', RUN_ROOT, '(ephemeral)' if not USE_GOOGLE_DRIVE else '(persistent)')

## Tests and configuration inspection

In [ ]:
RUN_TESTS = False
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest'], check=True)
subprocess.run([sys.executable, '-m', 'flowdistill_mri', 'inspect', '--config', 'configs/smoke_ci.yaml'], check=True)

## Opt-in CPU smoke
Set the flag only when you want to run the complete synthetic pipeline.

In [ ]:
RUN_CPU_SMOKE = False
if RUN_CPU_SMOKE:
    subprocess.run(['flowdistill-mri', 'smoke', '--tier', 'ci'], check=True)

## Teacher training and Track A distillation
Choose `colab_t4.yaml` or `colab_l4.yaml` after data support is enabled. Phase 0 uses synthetic configs.

In [ ]:
CONFIG = 'configs/smoke_local.yaml'
TEACHER_DIR = RUN_ROOT / 'teacher'
STUDENT_DIR = RUN_ROOT / 'freeflow'
RUN_TEACHER = False
RUN_DISTILLATION = False
if RUN_TEACHER:
    subprocess.run(['flowdistill-mri', 'train-teacher', '--config', CONFIG, '--output', str(TEACHER_DIR)], check=True)
if RUN_DISTILLATION:
    subprocess.run(['flowdistill-mri', 'distill-freeflow', '--config', CONFIG, '--teacher-checkpoint', str(TEACHER_DIR), '--output', str(STUDENT_DIR)], check=True)

## Inference and evaluation

In [ ]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    subprocess.run(['flowdistill-mri', 'evaluate-generation', '--config', CONFIG, '--teacher-checkpoint', str(TEACHER_DIR), '--student-checkpoint', str(STUDENT_DIR), '--output', str(RUN_ROOT / 'evaluation')], check=True)

## Track B — reserved
The arbitrary-state/two-time reconstruction interface is scaffolded, but its training remains deliberately disabled during initialization.